# ELLA VIDEO ENGINE — Wan 2.2 TI2V 5B\n\n**Kaggle GPU notebook | Text-to-Video + Image-to-Video**\n\n**Protected fast-start edition:** the original notebook is backed up in `notebooks/backups/`. A verified `sd-cli` binary is reused without running CMake or compiling again. New builds detect the actual GPU architecture, preserve the build directory, and compile only `sd-cli`. Set **Accelerator = GPU** and **Internet = On**. Kaggle session and GPU quotas still apply.\n

In [ ]:
import os, subprocess, shutil, json, pathlib, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert shutil.which('nvidia-smi'), 'Enable a Kaggle GPU accelerator in Notebook settings.'
required = {'git': 'git', 'cmake': 'cmake', 'g++': 'build-essential', 'ffmpeg': 'ffmpeg'}
missing = [pkg for cmd, pkg in required.items() if shutil.which(cmd) is None]
if missing:
    print('Installing missing system tools:', missing, flush=True)
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', *missing], check=True)
for module, package in [('huggingface_hub', 'huggingface_hub'), ('requests', 'requests')]:
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)
print('ELLA environment ready; Kaggle PyTorch/CUDA packages are reused.')

## 1. Minimal-source CUDA build (ELLA sd-cli)
This cell uses a shallow, blob-filtered **sparse checkout**: only the CMake files, headers, engine sources, CLI sources, and required vendored libraries are checked out. It does not check out assets, docs, WebUI/server, or unused media submodules. Only the `ggml` submodule is initialized. The CLI-only CMake file is narrowed locally to avoid configuring the unused server. A verified CUDA binary/build is reused when it matches the current GPU architecture.

In [ ]:
%cd /kaggle/working
import os, subprocess, shutil, pathlib, torch

REPO_DIR = pathlib.Path('/kaggle/working/stable-diffusion.cpp')
BUILD_DIR = REPO_DIR / 'build'
SDCLI_PATH = BUILD_DIR / 'bin' / 'sd-cli'
major, minor = torch.cuda.get_device_capability(0)
gpu_arch = f'{major}{minor}'
print('GPU:', torch.cuda.get_device_name(0), '| CUDA capability:', gpu_arch, flush=True)

def run(cmd, **kwargs):
    print('+', ' '.join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True, **kwargs)

# Keep an already-built, working CUDA CLI; do not download or rebuild it again.
cache_file = BUILD_DIR / 'CMakeCache.txt'
cache_text = cache_file.read_text(errors='ignore') if cache_file.is_file() else ''
cache_ok = ('SD_CUDA:BOOL=ON' in cache_text and
            f'CMAKE_CUDA_ARCHITECTURES:STRING={gpu_arch}' in cache_text)
binary_ok = SDCLI_PATH.is_file() and cache_ok
if binary_ok:
    try:
        p = subprocess.run([str(SDCLI_PATH), '--help'], capture_output=True,
                           text=True, timeout=30)
        binary_ok = p.returncode == 0
    except Exception:
        binary_ok = False

if not binary_ok:
    # Clone only Git metadata first; sparse checkout downloads blobs only for
    # the paths below, rather than checking out the repository's assets/docs.
    if not (REPO_DIR / '.git').is_dir():
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        run(['git', 'clone', '--depth=1', '--filter=blob:none',
             '--no-checkout', '--single-branch',
             'https://github.com/leejet/stable-diffusion.cpp.git', REPO_DIR])

    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError('ELLA: source checkout is not a valid Git repository.')

    run(['git', '-C', REPO_DIR, 'sparse-checkout', 'init', '--no-cone'])
    sparse_paths = [
        '/.gitmodules', '/LICENSE', '/CMakeLists.txt',
        '/cmake/', '/include/', '/src/', '/thirdparty/', '/ggml/',
        '/examples/CMakeLists.txt', '/examples/cli/', '/examples/common/'
    ]
    run(['git', '-C', REPO_DIR, 'sparse-checkout', 'set', '--no-cone', *sparse_paths])
    run(['git', '-C', REPO_DIR, 'checkout', '--force'])

    required_paths = [
        REPO_DIR / 'CMakeLists.txt',
        REPO_DIR / 'cmake' / 'ggml.cmake',
        REPO_DIR / 'src' / 'version.cpp',
        REPO_DIR / 'thirdparty' / 'CMakeLists.txt',
        REPO_DIR / 'examples' / 'cli' / 'CMakeLists.txt',
        REPO_DIR / '.gitmodules'
    ]
    missing = [str(p) for p in required_paths if not p.is_file()]
    if missing:
        raise RuntimeError('ELLA sparse checkout is incomplete: ' + ', '.join(missing))

    # The upstream examples CMake file also configures the unused web server.
    # Replace it only in this Kaggle working copy; keep the CLI target intact.
    (REPO_DIR / 'examples' / 'CMakeLists.txt').write_text(
        'add_subdirectory(cli)\n', encoding='utf-8')

    # ggml is the only required submodule. Do not initialize WebUI/WebP/WebM/CUTLASS.
    ggml_dir = REPO_DIR / 'ggml'
    if not (ggml_dir / 'CMakeLists.txt').is_file():
        run(['git', '-C', REPO_DIR, 'submodule', 'update',
             '--init', '--depth=1', '--', 'ggml'])
    if not (ggml_dir / 'CMakeLists.txt').is_file():
        raise RuntimeError('ELLA: required ggml submodule was not downloaded correctly.')

    # Work around Kaggle images where CMake cannot resolve CUDA::cuda_driver.
    CMAKE_FILE = REPO_DIR / 'CMakeLists.txt'
    cmake_text = CMAKE_FILE.read_text(encoding='utf-8')
    patch_marker = '# ELLA_KAGGLE_CUDA_DRIVER_TARGET'
    if patch_marker not in cmake_text:
        anchor = 'find_package(CUDAToolkit REQUIRED)'
        if anchor not in cmake_text:
            raise RuntimeError('ELLA: upstream CMake CUDA section changed; refusing unsafe patch.')
        driver_patch = r'''
# ELLA_KAGGLE_CUDA_DRIVER_TARGET
if(NOT TARGET CUDA::cuda_driver)
  find_library(ELLA_CUDA_DRIVER_LIBRARY NAMES cuda
    HINTS
      "${CUDAToolkit_LIBRARY_DIR}"
      "${CUDAToolkit_ROOT}/lib64"
      "${CUDAToolkit_ROOT}/targets/x86_64-linux/lib/stubs"
      "/usr/local/cuda/lib64/stubs"
      "/usr/local/cuda/targets/x86_64-linux/lib/stubs"
      "/usr/local/cuda/compat"
      "/usr/lib/x86_64-linux-gnu")
  if(NOT ELLA_CUDA_DRIVER_LIBRARY)
    find_file(ELLA_CUDA_DRIVER_LIBRARY NAMES libcuda.so.1
      PATHS "/usr/lib/x86_64-linux-gnu" "/usr/local/cuda/compat")
  endif()
  if(NOT ELLA_CUDA_DRIVER_LIBRARY)
    message(FATAL_ERROR "ELLA: CUDA driver library was not found.")
  endif()
  add_library(CUDA::cuda_driver UNKNOWN IMPORTED GLOBAL)
  set_target_properties(CUDA::cuda_driver PROPERTIES
    IMPORTED_LOCATION "${ELLA_CUDA_DRIVER_LIBRARY}")
endif()
'''
        CMAKE_FILE.write_text(cmake_text.replace(anchor, anchor + '\n' + driver_patch, 1),
                              encoding='utf-8')

    configure = [
        'cmake', '-S', str(REPO_DIR), '-B', str(BUILD_DIR),
        '-DCMAKE_BUILD_TYPE=Release', '-DSD_CUDA=ON',
        '-DSD_BUILD_EXAMPLES=ON', '-DSD_WEBP=OFF', '-DSD_WEBM=OFF',
        '-DSD_CUDA_CUTLASS_PATH:PATH=',
        f'-DCMAKE_CUDA_ARCHITECTURES={gpu_arch}'
    ]
    run(configure)

    mem_kb = 0
    for line in pathlib.Path('/proc/meminfo').read_text().splitlines():
        if line.startswith('MemAvailable:'):
            mem_kb = int(line.split()[1])
            break
    jobs = min(os.cpu_count() or 2, 4)
    if mem_kb and mem_kb < 12 * 1024 * 1024:
        jobs = min(jobs, 2)

    print(f'ELLA: compiling only sd-cli with {jobs} jobs; build cache retained.', flush=True)
    run(['cmake', '--build', BUILD_DIR, '--config', 'Release',
         '--target', 'sd-cli', '--parallel', str(jobs)])

    if not SDCLI_PATH.is_file():
        raise RuntimeError(f'ELLA: build completed but sd-cli is missing: {SDCLI_PATH}')
    check = subprocess.run([str(SDCLI_PATH), '--help'], capture_output=True,
                           text=True, timeout=30)
    if check.returncode != 0:
        raise RuntimeError('ELLA: newly built sd-cli failed its --help smoke test: ' +
                           (check.stderr[-3000:] if check.stderr else check.stdout[-3000:]))

SDCLI = str(SDCLI_PATH)
print('ELLA CUDA CLI VERIFIED:', SDCLI, flush=True)


## 2. Download the model components (Q4)
The generator alone is not enough: Wan also requires its matching VAE and UMT5 text encoder.

In [ ]:
from huggingface_hub import hf_hub_download
MODEL_DIR = pathlib.Path('/kaggle/working/ella_models'); MODEL_DIR.mkdir(exist_ok=True)
def get_file(repo, name):
    target = MODEL_DIR / name
    if not target.exists():
        hf_hub_download(repo_id=repo, filename=name, local_dir=str(MODEL_DIR))
    assert target.exists(), f'Missing download: {target}'
    print(name, round(target.stat().st_size/1e9, 2), 'GB')
get_file('off-grid-ai/Wan2.2-TI2V-5B-GGUF', 'Wan2.2-TI2V-5B-Q4_K_S.gguf')
get_file('off-grid-ai/Wan2.2-TI2V-5B-GGUF', 'wan2.2_vae.safetensors')
get_file('off-grid-ai/Video-Companions-GGUF', 'umt5-xxl-encoder-Q4_K_M.gguf')


## 3. T4 validation profile

The manual test below is five seconds only as a quick smoke test. The connected worker preserves the requested scene duration and continues longer scenes in five-second chunks.

In [ ]:
PROMPT = 'A cinematic realistic scene: a young woman walks slowly through a rainy Istanbul street at night, reflections on the pavement, natural human movement, dramatic film lighting, smooth camera tracking.'
NEGATIVE_PROMPT = 'low quality, blurry, distorted anatomy, extra fingers, flicker, subtitles, watermark, static frame'
MODE = 'text'  # use 'image' for image-to-video
REFERENCE_IMAGE = '/kaggle/working/reference.png'
DURATION_SECONDS = 5
FPS = 8
WIDTH, HEIGHT = 832, 480
STEPS = 8
SEED = 42
print('Fast validation settings ready:', WIDTH, 'x', HEIGHT, '|', STEPS, 'steps |', DURATION_SECONDS, 'seconds')


In [ ]:
!mkdir -p /kaggle/working/ella_engine
print('ELLA runner will be fetched from the project branch in the next cell.')


## 4. Generate MP4
This cell calls the ELLA runner with argument-safe subprocess execution (no shell interpolation).

In [ ]:
import importlib.util
import urllib.request
RUNNER_PATH = pathlib.Path('/kaggle/working/ella_engine/runner.py')
RUNNER_PATH.parent.mkdir(parents=True, exist_ok=True)
url='https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/ella-wan22-engine/services/ella-wan-engine/runner.py'
# Always refresh the runner so Kaggle cannot reuse an older cached copy.
urllib.request.urlretrieve(url, RUNNER_PATH)
spec=importlib.util.spec_from_file_location('ella_runner', RUNNER_PATH)
runner=importlib.util.module_from_spec(spec); spec.loader.exec_module(runner)
args=runner.parser().parse_args(['--prompt', PROMPT, '--negative-prompt', NEGATIVE_PROMPT, '--duration', str(DURATION_SECONDS), '--fps', str(FPS), '--width', str(WIDTH), '--height', str(HEIGHT), '--steps', str(STEPS), '--seed', str(SEED), '--output', '/kaggle/working/ella_output.mp4'])
args.sd_cli=pathlib.Path(SDCLI)
args.diffusion_model=MODEL_DIR/'Wan2.2-TI2V-5B-Q4_K_S.gguf'
args.vae=MODEL_DIR/'wan2.2_vae.safetensors'
args.text_encoder=MODEL_DIR/'umt5-xxl-encoder-Q4_K_M.gguf'
if MODE == 'image':
    args.image=pathlib.Path(REFERENCE_IMAGE)
result=runner.run(args)
print(json.dumps(result, ensure_ascii=False, indent=2))
from IPython.display import Video, display
display(Video('/kaggle/working/ella_output.mp4', embed=True))


## 5. Download your MP4
The final file is `/kaggle/working/ella_output.mp4`. Use the Kaggle file browser to download it. Keep the generation log (`ella_output.log`) if anything fails.

In [ ]:
!ls -lh /kaggle/working/ella_output.mp4 /kaggle/working/ella_output.log


## 6. Connect the worker to Ella2027

Run this cell after the manual MP4 test succeeds. Use the existing Kaggle Secret `ELLA_KAGGLE_WORKER_TOKEN`, with the same value as Render `KAGGLE_LTX_WORKER_TOKEN` and the ELLA provider key. The worker automatically continues scenes longer than five seconds in sequential GPU-safe chunks.

In [ ]:
import urllib.request, sys
WORKER_PATH = pathlib.Path('/kaggle/working/ella_engine/wan22_worker.py')
WORKER_PATH.parent.mkdir(parents=True, exist_ok=True)
WORKER_URL = 'https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/ella-wan22-engine/services/kaggle-wan22-worker/wan22_worker.py'
urllib.request.urlretrieve(WORKER_URL, WORKER_PATH)

required = [
    pathlib.Path(RUNNER_PATH), pathlib.Path(SDCLI),
    pathlib.Path(MODEL_DIR) / 'Wan2.2-TI2V-5B-Q4_K_S.gguf',
    pathlib.Path(MODEL_DIR) / 'wan2.2_vae.safetensors',
    pathlib.Path(MODEL_DIR) / 'umt5-xxl-encoder-Q4_K_M.gguf',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing ELLA runtime files: ' + ', '.join(missing))

from kaggle_secrets import UserSecretsClient
ELLA_TOKEN = UserSecretsClient().get_secret('ELLA_KAGGLE_WORKER_TOKEN')
if not ELLA_TOKEN:
    raise ValueError('Kaggle Secret ELLA_KAGGLE_WORKER_TOKEN is empty.')

sys.path.insert(0, str(WORKER_PATH.parent))
from wan22_worker import run_forever
print('[ELLA] Wan 2.2 worker starting. Requested scene lengths are split into GPU-safe continuation chunks.', flush=True)
run_forever(
    'https://ella2027.onrender.com', ELLA_TOKEN,
    pathlib.Path(RUNNER_PATH), pathlib.Path(MODEL_DIR),
    pathlib.Path(SDCLI), pathlib.Path('/kaggle/working/ella_jobs'),
    max_scene_seconds=5, idle_seconds=12
)